In [ ]:
# variable was changed to the particle number density and the total liquid content

import sys
import numpy as np
import math
import matplotlib.pyplot as plt
from scipy import signal, interpolate
from matplotlib import pylab as plt
from scipy.integrate import odeint
from scipy.integrate import quad
from scipy.integrate import solve_ivp

#preexisting particles
Npre = 200.0E6   # 40 /cm3 (Knollenberg&Hunten 1980, Table 4)
rpre = 0.3E-6 / 2.  # mode diameter = 0.3 micron (Knollenberg&Hunten 1980, Table 4)
# rpre = 0.2E-6 / 2.  # mode diameter = 0.3 micron (Knollenberg&Hunten 1980, Table 4)
Spre = 4.*np.pi*rpre*rpre * Npre  # total surface area per cubic meters

print("Spre=",Spre)

#Hamill_etal1977JAS  Table 2  (for 80% H2SO4)
Tension = 80.0E-7 * 1.0E4  #J/m2 #80 erg/cm2 = dyn/cm
R = 8.314 # J/K/mol
M = 0.098 # kg/mol　　# 98.079 g/mol
Rho_l = 1.83E-3 * 1.0E6 # kg/m3 # 1.83 g/cm3
Boltz = 1.38064852E-23 # m2 kg s-2 K-1 
NA = 6.02214076E23 # mol-1
MieFactor = 2.0

# VIRA: Seiff 1985
R_CO2 = 191.4
Cp_CO2 = 904  #@50km

# initial particle number density (1/m3)
Ninit = 0

Oneday = 86400.

# JUMP parameters
hjump = 3000.
taujump = 1200
tjump = 60*200

# background speed
c = 15

# cloud and H2SO4 distributions
Zcloudbase = 47.5e3
fmiddlecloud = 0 
Hupper = 10.0e3
Hlower = 4.0e3 
THICKNESS = 0

# top height of cloud thickness calculation
Column_top = 47.5e3

# altitude range
Zbottom = 44.0e3 #46e3
Ztop =48.0e3

# altitde interval
dz = 100.

# altitude for output
ZSAVE = 46.5e3

# CCNs loss
nc_decrease_fac = 0

# adiabatic lapse rate
Gd = 9.76e-3 # K/m @ 50km

# gravitational acceleration
GACC=8.67                   #m/s^2 [VIRA]

In [ ]:
# readig VIRA

ZVIRA = np.array([])
TVIRA = np.array([])
PVIRA = np.array([])
RHOVIRA = np.array([])

# Altitude (km) Temperature (K)	pressure (bars)	density (kg m-3)
ZVIRA,TVIRA,PVIRA ,RHOVIRA = np.loadtxt('./VIRA_noheader.csv',   delimiter=',', unpack=True, usecols=[0, 1, 2, 3])
NVIRA = len(ZVIRA)
print("NVIRA=",NVIRA)
ZVIRA = ZVIRA* 1000.
PVIRA = PVIRA * 1.0e5

N2VIRA = np.zeros(NVIRA) 
for i in range (1,NVIRA-1,1):
    N2VIRA[i] = GACC/TVIRA[i] * ((TVIRA[i+1]-TVIRA[i-1])/(ZVIRA[i+1]-ZVIRA[i-1]) + Gd)

#coordinate
zgrid = np.arange(Zbottom, Ztop+dz, dz)
Nz = len(zgrid)
print('alt levels=',Nz)
print('zgrid=',zgrid)

##altitude level for output
IZSAVE = 0
for iz in range(Nz):
    if np.abs(ZSAVE-zgrid[iz]) < 0.01:
        IZSAVE = iz
print("ZSAVE= ", ZSAVE, "  IZSAVE= ",IZSAVE, " zgrid(IZSAVE)= ",zgrid[IZSAVE])

P0 = np.array([])
fn = interpolate.interp1d(ZVIRA, PVIRA)
for iz in range(Nz):
    P0 = np.append(P0, fn(zgrid[iz]))

T0 = np.array([])
fn = interpolate.interp1d(ZVIRA, TVIRA)
for iz in range(Nz):
    T0 = np.append(T0, fn(zgrid[iz]))

RHO0 = np.array([])
fn = interpolate.interp1d(ZVIRA, RHOVIRA)
for iz in range(Nz):
    RHO0 = np.append(RHO0, fn(zgrid[iz]))


In [ ]:
#total temperature
def temperature(time):
    # phase = omega * time
    z = zgrid[iz]
    phi1, u1, w1, xi1, zeta1, temp1 = jump(z, time, hjump, tjump, taujump)
    T = temp1 + Temp0

    return T

In [ ]:
#meteorological variabls during jump
def jump(z, time, hjump, tjump, taujump):    # z (m), phase (rad), uamp (m/s), k (1/m)     
    zeta = hjump/2 * (1. + np.tanh((time-tjump)/(taujump/2)))
    phi = 0
    u = 0
    w = 0
    xi = 0
    temp = -(GACC/Cp_CO2) * zeta
    
    return phi, u, w, xi, zeta, temp

In [ ]:
#saturation vapor pressure of H2SO4
def psat_H2SO4(temp):
    T00 = 385.0
    return np.exp( 16.259 - 10156.0 / temp + 7.42 * ( 1.0 + np.log( T00 / temp ) - T00 / temp ) ) * 1.0e5

In [ ]:
#saturation profile
def satmixratio_at_z(z):
    fn = interpolate.interp1d(zgrid, T0)
    psat = psat_H2SO4(fn(z))
    fn = interpolate.interp1d(zgrid, P0)
    p = fn(z)
    f_gas = psat/p
    return f_gas

In [ ]:
#background mixing ratio
def mixingratio_basic(z):
    # H2SO4 vapor
    if (z > Ztop):
        mixratio = 0
    elif (z > Zcloudbase):
        mixratio = satmixratio_at_z(z)
    elif (z > Zcloudbase-THICKNESS):
        mixratio = satmixratio_at_z(Zcloudbase)
    else:
        mixratio = satmixratio_at_z(Zcloudbase) * np.exp((z-(Zcloudbase-THICKNESS))/Hlower)
    
    return mixratio

In [ ]:
#plot background mixing ratio
mixrat = np.array([])
for iz in range(Nz):
    mixrat  = np.append(mixrat, mixingratio_basic(zgrid[iz]))

plt.plot(zgrid, mixrat,'.')
plt.show()

In [ ]:
#critical radius
def criticalradius(satratio, temp):
    if satratio > 1.0:
        r = 2.0 * M * Tension / (R * temp * Rho_l * np.log(satratio))
    else:
        r = 0
    
    return r

In [ ]:
#free energy F
def freeenergy(satratio, temp):
    F = 16 * np.pi * M**2 * Tension**3 / 3 / (R * temp * Rho_l * np.log(satratio))**2
        
    return F * freeenergy_fac

In [ ]:
#nucleation rate
def nucleationrate(satratio, temp):

    if satratio > 1:
        # Heterogeneous nucleation (Fletcher 1958)
        J = 1.0E25 * 1.0E4 * Spre * np.exp(-freeenergy(satratio, temp)/Boltz/temp)
        # Homogeneous nucleation (Zeifman2005, Hamill2009)
#         J = 1/Rho_l * (2 * Tension * M/NA /np.pi)**0.5  *  (psat_H2SO4(temp)*satratio/Boltz/temp)**2  * np.exp(-freeenergy(satratio, temp)/Boltz/temp)
    else:
        J = 0
        
    return J * nucreate_fac

In [ ]:
#saturation vapor pressure over droplets
def psat_H2SO4_sphere(temp,r):
    psat = psat_H2SO4(temp) * np.exp(2 * Tension * M / (R * temp * r * Rho_l))
    return psat

In [ ]:
#diffusion coefficient
def diff(temp):  #Pruppacher, p.503, for water vapor in air
    Dv = 0.211E-4 * (temp/273.15)**1.94 * (1.01325E5/pressure(temp))
    return Dv

In [ ]:
#mass change per particle
def dropletgrowth(p_H2SO4, r, temp):  #Pruppacher, p.504
    if r > 0:
        dmdt = 4 * np.pi * r * diff(temp) * M/R * (p_H2SO4-psat_H2SO4_sphere(temp,r))/temp # m m2 s-1 kg K J-1 J m-1 m-2 K-1 = kg s-1
    else:
        dmdt = 0

    return dmdt

In [ ]:
#pressure change in adiabatic process
def pressure(temp): 
    p = Press * (temp/Temp0)**(Cp_CO2/R_CO2)
    return p

In [ ]:
#air parcel volume change
def expansion(temp): 
    ex = Press/pressure(temp) * temp/Temp0
    # ex = 1
    return ex

In [ ]:
#particle mass change including all sizes
def func_totalgrowth(lnr,p_H2SO4,temp,A,B):
    r = np.exp(lnr)
    f = dropletgrowth(p_H2SO4, r, temp) * A * r**2 * np.exp(-B*r) * r
    return f

In [ ]:
#differential equation
def func(time, state):
# def func(state, time): #odeint
    nc, total_l = state

    # ambient vapor pressure and saturation ratio
    temp = temperature(time)
    p_H2SO4 = (Rho_g0 - total_l) * (R/M) * temp / expansion(temp) #air parcel volume change  
    satratio = p_H2SO4 / psat_H2SO4(temp)
    
    # particle number density change
    dncdt1 = nucleationrate(satratio, temp)
    
    # particle mass change due to nucleation
    dtotal_ldt1 = 4.0/3.0*np.pi*criticalradius(satratio, temp)**3*Rho_l*dncdt1 #20230707 *expansion(temp)##20230611??? 

# mean particle size
    flgmeanradius=0
    if nc <= 0 or total_l <0:
        meanradius = 0
    else :
        flgmeanradius = 1
        meanradius = ( total_l / (4.0/3.0*np.pi*nc*Rho_l) )**(1.0/3.0)
    
# particle mass change due to condensation/evaporation
    dtotal_ldt2 = 0
    if nc > 0 and total_l > 0:
        V = total_l / Rho_l
        B = (80.0*np.pi*nc/V)**(1.0/3.0)
        A = B**3 * nc / 2.0
        
        if Gamma == 0:
            dtotal_ldt2 = nc * dropletgrowth(p_H2SO4, meanradius, temp)
        elif Gamma == 1:
            rr =  np.array([])
            dintg = np.array([])
            growth = np.array([])
            expfac = np.array([])
            rfac = np.array([])
            totalgrowth = 0.
            for i in range(nlnr):
                lnr = lnrmin + i*dlnr
                r = np.exp(lnr)
                totalgrowth += dropletgrowth(p_H2SO4, r, temp) * A * r**2 * np.exp(-B*r) * r * dlnr
                ### for check
                dintg = np.append(dintg, dropletgrowth(p_H2SO4, r, temp) * A * r**2 * np.exp(-B*r) * r )
                rr = np.append(rr,r)
                growth = np.append(growth,dropletgrowth(p_H2SO4, r, temp))
                expfac = np.append(expfac,np.exp(-B*r))
                rfac = np.append(rfac,A * r**2 * r)
                ###
            dtotal_ldt2 = totalgrowth     
        elif Gamma == 2:
            dtotal_ldt2 = quad(func_totalgrowth, lnrmin, lnrmax, args=(p_H2SO4,temp,A,B))[0]

# particle number density change due to evaporation
    dncdt2 = 0
    evap=0
    if dtotal_ldt2 < 0:
        evap=1
        dncdt2 = dtotal_ldt2 / (4.0/3.0*np.pi*meanradius**3*Rho_l)  * nc_decrease_fac
        
# sum of contributions from nucleation and condensation
    dncdt = dncdt1 + dncdt2
    dtotal_ldt = dtotal_ldt1 + dtotal_ldt2
    
    return [dncdt,  dtotal_ldt]


In [ ]:
# scattering cross section
def func_crosssection(lnr,A,B):
    r = np.exp(lnr)
    f = MieFactor * np.pi*r**2 * A * r**2 * np.exp(-B*r) * r
    return f

In [ ]:
def plot_timeseries():
    
    SIZEX = 6
    SIZEY = 2
    
    x_list_show = np.array([])
    z_list_show = np.array([])
    
    time_list_show = state_list.t
    temp_list = temperature(time_list_show)
    state_list_show = state_list.y
    
    ## air parcel expansion
    state_list_show[0, :] = state_list_show[0, :] / expansion(temp_list)  
    state_list_show[1, :] = state_list_show[1, :] / expansion(temp_list) 
    
    for i in range(len(time_list_show)):
        phi1, u1, w1, xi1, zeta1, temp1 = jump(zgrid[iz], time_list_show[i], hjump, tjump, taujump)
        
        x_list_show = np.append(x_list_show, c*time_list_show[i])  #time series --> horizontal distance

        z_list_show = np.append(z_list_show, zgrid[iz]+zeta1)
    xmax = x_list_show[len(x_list_show)-1]
    
    FONTSIZE = 11
    LWIDTH = 1
    
##Temperature
    # plt.figure(figsize=(SIZEX,SIZEY))
    # plt.tick_params(labelsize=FONTSIZE)
    # plt.xlabel("Time from start (minutes)", fontsize=FONTSIZE)
    # plt.ylabel("Temperature (K)", fontsize=FONTSIZE)
    # plt.plot(time_list_show/60, temp_list,'-',linewidth=LWIDTH,color="black")
    # plt.xlim([tstart/60, tend/60])
    # plt.ylim([345,385])
    if iz==IZSAVE:
        print("Save data at ",zgrid[iz]," km")
        with open('temperature'+PARAM+'.txt', 'w') as f:
            for i in range(len(time_list_show)):
                print(time_list_show[i]/60,'  ',temp_list[i], file=f)
        f.close()
    # plt.show()

    p_H2SO4_list = (Rho_g0/expansion(temp_list)-state_list_show[1, :]) * (R/M) * temp_list ##20230611??? 

##Saturation ratio
    satratio_list = p_H2SO4_list / psat_H2SO4(temp_list) 
    # plt.figure(figsize=(SIZEX,SIZEY))
    # plt.tick_params(labelsize=FONTSIZE)
    # plt.xlabel("Time from start (minutes)", fontsize=FONTSIZE)
    # plt.ylabel("Saturation ratio", fontsize=FONTSIZE)
    # plt.plot(time_list_show/60, satratio_list,'-',linewidth=LWIDTH,color="black")
    # plt.xlim([tstart/60, tend/60])
    if iz == IZSAVE:
        with open('satratio'+PARAM+'.txt', 'w') as f:
            for i in range(len(time_list_show)):
                print(time_list_show[i]/60,'  ',satratio_list[i], file=f)
        f.close()
    # plt.show()

##Number density
    # plt.figure(figsize=(SIZEX,SIZEY))
    # plt.tick_params(labelsize=FONTSIZE)
    # plt.xlabel("Time from start (minutes)", fontsize=FONTSIZE)
    # plt.ylabel("Number density (cm$^{-3}$)", fontsize=FONTSIZE)
    # plt.plot(time_list_show / 60, state_list_show[0, :]/1.0E6,'-',linewidth=LWIDTH,color="black")
    # plt.xlim([tstart/60, tend/60])
    if iz == IZSAVE:
        with open('numden'+PARAM+'.txt', 'w') as f:
            for i in range(len(time_list_show)):
                print(time_list_show[i]/60,'  ',state_list_show[0,i]/1.0E6, file=f)
        f.close()
    # plt.show()

##Mean radius
    r_list = np.array([])
    rcrit_list = np.array([])
    # plt.figure(figsize=(SIZEX,SIZEY))
    # plt.tick_params(labelsize=FONTSIZE)
    # plt.xlabel("Time from start (minutes)", fontsize=FONTSIZE)
    # plt.ylabel("Mean radius ($\mu$m)", fontsize=FONTSIZE)
    for i in range(len(time_list_show)):
        rcrit_list = np.append(rcrit_list, criticalradius(satratio_list[i], temp_list[i]))
        if (state_list_show[0, i] <= 0 or state_list_show[1, i] < 0):
            r_list = np.append(r_list, 0)
        else:
            r_list = np.append(r_list,  ( state_list_show[1, i] / (4.0/3.0*np.pi*state_list_show[0, i]*Rho_l) )**(1.0/3.0) )
    # plt.plot(time_list_show/60, r_list*1e6 ,'-',linewidth=LWIDTH,color="black")
    # plt.xlim([tstart/60, tend/60])
    if iz == IZSAVE:
        # plt.savefig('radius.pdf', bbox_inches='tight')
        with open('radius'+PARAM+'.txt', 'w') as f:
            for i in range(len(time_list_show)):
                print(time_list_show[i]/60,'  ',r_list[i]*1e6, file=f)
        f.close()
    # plt.show()
    
    for i in range(len(time_list_show)):
        if state_list_show[0,i] > 0 and state_list_show[1,i] > 0:
            V = state_list_show[1,i] / Rho_l
            B = (80.0*np.pi*state_list_show[0,i]/V)**(1.0/3.0)
            A = B**3 * state_list_show[0,i] / 2.0
            opacity = quad(func_crosssection, lnrmin, lnrmax, args=(A,B))[0]
        else:
            opacity = 0
        c_list_show2d[iz][i] = opacity  # m2 * m-3 = m-1
    
    # plt.figure(figsize=(SIZEX*1.2,SIZEY))
    # plt.tick_params(labelsize=FONTSIZE)
    # plt.xlabel("Eastward distance (km)", fontsize=FONTSIZE)
    # plt.ylabel("Altitude (km)", fontsize=FONTSIZE)
    # MarkSize = 1.2
    # plt.xlim([0,360])
    # plt.ylim([42,52])
    # cm = plt.cm.get_cmap('binary')
    # plt.scatter(x_list_show/1000, z_list_show/1000, vmin=0, vmax=5, c=c_list_show2d[iz,:]*1000, cmap='plasma_r', s=MarkSize)
    # cbar = plt.colorbar() # カラーバーを描く
    # cbar.set_label('Extinction coefficient (km$^{-1}$)')
    if iz == IZSAVE:
        plt.savefig('opacity'+PARAM+'.pdf', bbox_inches='tight')
    # plt.show()
    
    cnum = state_list_show[0,:]
    for i in range(len(time_list_show)):
        x_list_show2d[iz][i] = x_list_show[i]
        z_list_show2d[iz][i] = z_list_show[i]
    

In [ ]:
# vertical integration
def plot_column():
    
    # x-coordinate
    dx = 2e2
    xgrid = np.arange(0, np.max(x_list_show2d[0]), dx)
    Nx = len(xgrid)

    x_mean = np.zeros((Nz,Nx))
    z_mean = np.zeros((Nz,Nx))
    c_mean = np.zeros((Nz,Nx))

    for iz in range(Nz):
        it = 0
        flag = False
        for ix in range(Nx-1):
            n_mean = 0
            while x_list_show2d[iz][it] < xgrid[ix+1]:
                x_mean[iz][ix] += x_list_show2d[iz][it]
                z_mean[iz][ix] += z_list_show2d[iz][it]
                c_mean[iz][ix] += c_list_show2d[iz][it]    # 1/m
                n_mean += 1
                it += 1
                if (it >= len(state_list.t)-1): 
                    flag = True
                    break
            if (n_mean != 0):
                x_mean[iz][ix] /= n_mean
                z_mean[iz][ix] /= n_mean
                c_mean[iz][ix] /= n_mean   # 1/m
                n_mean = 0
            if flag:  #20231221
                break

    # vertical integration            
    c_column = np.zeros(Nx)
    for ix in range(Nx-1):
        c_column[ix] = 0
        for iz in range(0,Nz):
            if zgrid[iz] <= Column_top:
                c_column[ix] += c_mean[iz][ix] * dz

    # plot column
    MarkSize = 2
    SIZEX = 6
    SIZEY = 2
    
    FONTSIZE = 11
    LWIDTH = 1
    
    with open('opticalthickness'+PARAM+'.txt', 'w') as f:
        for i in range(Nx-1):
            print(xgrid[i]/1000,'  ',c_column[i], file=f)
    f.close()

    # opacity 2D plot
    cmax = np.max(c_mean)
    print("cmax = ",cmax)
    if cmax > 0.001:
        cmax = 0.001

    cm = plt.cm.get_cmap('binary')
    
    MarkSize = 0.1

    plt.figure(figsize=(SIZEX*1.2,SIZEY))
    plt.xlim([0,360])
    plt.ylim([42,52])
    plt.xlabel("Eastward distance (km)", fontsize=FONTSIZE)
    plt.ylabel("Altitude (km)", fontsize=FONTSIZE)
    for iz in reversed(range(Nz)):
        if zgrid[iz] <= Column_top:
            plt.scatter(x_mean[iz]/1000, z_mean[iz]/1000, c=c_mean[iz]*1000, vmin=0, vmax=5, cmap='plasma_r', s=MarkSize)
    plt.tick_params(labelsize=FONTSIZE)
    cbar = plt.colorbar()
    cbar.set_label('Extinction coefficient (km$^{-1}$)')
    plt.savefig('opacity2D'+PARAM+'.pdf', bbox_inches='tight')    # 1/km
    plt.show()


In [ ]:
# time integration

TimeEnd = tjump * 2 
t_span = [0, TimeEnd]
TimeStep = 10
Nstep = int(TimeEnd / TimeStep)
dt = TimeStep
tstart = 0
tend = TimeEnd
time_list = np.linspace(0.0, TimeEnd, Nstep)

Gamma = 2

rmin = 0.05E-6
rmax = 50.0E-6
lnrmin = np.log(rmin)
lnrmax = np.log(rmax)

dlnr = 0.1
nlnr = int((lnrmax-lnrmin)/dlnr)
# print("nlnr=",nlnr)

useodeint = 0

nucreate_fac = 1

flg = 0

for freeenergy_fac in [0.0003, 0.003]:
    print("freeenergy_fac=", freeenergy_fac)

    PARAM = str(freeenergy_fac)

    for iz in range(Nz):
        if zgrid[iz] <= Column_top:
            print("z=", zgrid[iz])

            # initial condition
            Press = P0[iz]
            Temp0 = T0[iz]
            rho0 = RHO0[iz]
            MixRatio = mixingratio_basic(zgrid[iz])

            Rho_g0 = Press * MixRatio / (R/M) / Temp0   # initial H2SO4 gas density
            state = [Ninit, 0]
            state0 = state

            # solve ordinary differential equation
            state_list = solve_ivp(func, t_span, state0, method='RK45',t_eval=time_list)

            # defining arrays for vertical profile plot
            if (flg==0):
                Nt = len(time_list)
                x_list_show2d = np.zeros((Nz,Nt))
                z_list_show2d = np.zeros((Nz,Nt))
                c_list_show2d = np.zeros((Nz,Nt))
                flg = 1

            plot_timeseries()


    plot_column()


In [ ]:
def plot_column_from_file():

    # plot column
    MarkSize = 2
    SIZEX = 6
    SIZEY = 2
    
    FONTSIZE = 11
    LWIDTH = 1
    
    ## optical thickness plot
    file=open('opticalthickness'+PARAM1+'.txt')
    lines=file.readlines()
    file.close()
    x1=[ ]
    y1=[ ]
    for i in range(0,len(lines),1):
        x,y=map(float,lines[i].split())
        # print("i=",i," x=",x," y=",y)
        x1.append([x])
        y1.append([y])
    file=open('opticalthickness'+PARAM2+'.txt')
    lines=file.readlines()
    file.close()
    x2=[ ]
    y2=[ ]
    for i in range(0,len(lines),1):
        x,y=map(float,lines[i].split())
        # print("i=",i," x=",x," y=",y)
        x2.append([x])
        y2.append([y])
    
    plt.figure(figsize=(SIZEX,SIZEY))
    plt.tick_params(labelsize=FONTSIZE)
    plt.xlabel("Eastward distance (km)", fontsize=FONTSIZE)
    plt.ylabel("Optical thickness", fontsize=FONTSIZE)
    plt.xlim([0,360])
    plt.ylim([0,25])
    # plt.plot(xgrid/1000, c_column,'o',linewidth=LWIDTH,color="black")
    plt.plot(x1, y1,'-',linewidth=LWIDTH,color="black")
    plt.plot(x2, y2,'-',linewidth=LWIDTH,color="red")
    plt.savefig('optthick.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
def plot_timeseries_from_file():
    
    SIZEX = 6
    SIZEY = 2    
    
    FONTSIZE = 11
    LWIDTH = 1
    
    XMIN=0
    XMAX=400
    
    
##Temperature
    file=open('temperature'+PARAM1+'.txt')
    lines=file.readlines()
    file.close()
    x1=[ ]
    y1=[ ]
    for i in range(0,len(lines),1):
        x,y=map(float,lines[i].split())
        x1.append([x])
        y1.append([y])
    file=open('temperature'+PARAM2+'.txt')
    lines=file.readlines()
    file.close()
    x2=[ ]
    y2=[ ]
    for i in range(0,len(lines),1):
        x,y=map(float,lines[i].split())
        x2.append([x])
        y2.append([y])
    plt.figure(figsize=(SIZEX,SIZEY))
    plt.tick_params(labelsize=FONTSIZE)
    plt.xlim(XMIN,XMAX)
    plt.ylim([340,385])
    plt.xlabel("Time from start (minutes)", fontsize=FONTSIZE)
    plt.ylabel("Temperature (K)", fontsize=FONTSIZE)
    plt.plot(x1, y1,'-',linewidth=LWIDTH,color="black")
    # plt.plot(x2, y2,'--',linewidth=LWIDTH,color="blue")
    plt.savefig('temperature.pdf', bbox_inches='tight')
    plt.show()

##Saturation ratio
    file=open('satratio'+PARAM1+'.txt')
    lines=file.readlines()
    file.close()
    x1=[ ]
    y1=[ ]
    for i in range(0,len(lines),1):
        x,y=map(float,lines[i].split())
        x1.append([x])
        y1.append([y])
    file=open('satratio'+PARAM2+'.txt')
    lines=file.readlines()
    file.close()
    x2=[ ]
    y2=[ ]
    for i in range(0,len(lines),1):
        x,y=map(float,lines[i].split())
        x2.append([x])
        y2.append([y])
    plt.figure(figsize=(SIZEX,SIZEY))
    plt.tick_params(labelsize=FONTSIZE)
    plt.xlim(XMIN,XMAX)
    plt.ylim([0.5,1.4])
    plt.xlabel("Time from start (minutes)", fontsize=FONTSIZE)
    plt.ylabel("Saturation ratio", fontsize=FONTSIZE)
    plt.plot(x1, y1,'-',linewidth=LWIDTH,color="black")
    plt.plot(x2, y2,'-',linewidth=LWIDTH,color="red")
    plt.savefig('satratio.pdf', bbox_inches='tight')
    plt.show()

##Number density
    file=open('numden'+PARAM1+'.txt')
    lines=file.readlines()
    file.close()
    x1=[ ]
    y1=[ ]
    for i in range(0,len(lines),1):
        x,y=map(float,lines[i].split())
        x1.append([x])
        y1.append([y])
    file=open('numden'+PARAM2+'.txt')
    lines=file.readlines()
    file.close()
    x2=[ ]
    y2=[ ]
    for i in range(0,len(lines),1):
        x,y=map(float,lines[i].split())
        x2.append([x])
        y2.append([y])
    plt.figure(figsize=(SIZEX,SIZEY))
    plt.tick_params(labelsize=FONTSIZE)
    plt.xlim(XMIN,XMAX)
    plt.ylim([10,10000])
    plt.yscale('log')
    plt.xlabel("Time from start (minutes)", fontsize=FONTSIZE)
    plt.ylabel("Number density (cm$^{-3}$)", fontsize=FONTSIZE)
    plt.plot(x1, y1,'-',linewidth=LWIDTH,color="black")
    plt.plot(x2, y2,'-',linewidth=LWIDTH,color="red")
    plt.savefig('numden.pdf', bbox_inches='tight')
    plt.show()

##Mean radius
    file=open('radius'+PARAM1+'.txt')
    lines=file.readlines()
    file.close()
    x1=[ ]
    y1=[ ]
    for i in range(0,len(lines),1):
        x,y=map(float,lines[i].split())
        x1.append([x])
        y1.append([y])
    file=open('radius'+PARAM2+'.txt')
    lines=file.readlines()
    file.close()
    x2=[ ]
    y2=[ ]
    for i in range(0,len(lines),1):
        x,y=map(float,lines[i].split())
        x2.append([x])
        y2.append([y])
    plt.figure(figsize=(SIZEX,SIZEY))
    plt.tick_params(labelsize=FONTSIZE)
    plt.xlim(XMIN,XMAX)
    plt.ylim([0,4])
    plt.xlabel("Time from start (minutes)", fontsize=FONTSIZE)
    plt.ylabel("Mean radius ($\mu$m)", fontsize=FONTSIZE)
    plt.plot(x1, y1,'-',linewidth=LWIDTH,color="black")
    plt.plot(x2, y2,'-',linewidth=LWIDTH,color="red")
    plt.savefig('radius.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
PARAM1 = '0.0003'
PARAM2 = '0.003'
    
plot_timeseries_from_file()
plot_column_from_file()